# Hisn al-Muslim two-source merge notebook

This notebook builds a merged Hisn al-Muslim dataset from two sources:

- **Sunnah.com** for:
  - `dua_id`
  - English translation
  - English transliteration
  - Arabic
  - English/Arabic chapter metadata

- **HadithBD** for:
  - `dua_id`
  - Bengali translation
  - Arabic
  - Bengali transliteration if available
  - Bengali chapter metadata

## Merge strategy

Do **not** join on chapter count alone.  
Join on the **individual dua number** (`dua_id`).

Then validate Arabic from both sources:

- exact normalized match
- near-exact match
- close match
- mismatch
- one side missing
- both missing

## Outputs

The notebook writes:

- `sunnah_hisn_raw.json`
- `hadithbd_hisn_raw.json`
- `hisn_merged.csv`
- `hisn_merged.json`
- `hisn_arabic_audit.csv`


In [1]:
!python --version

Python 3.11.5


In [2]:
# Install if needed
# !pip install -U requests certifi truststore beautifulsoup4 lxml pandas
# Optional if you need rendered HTML fallback:
# !pip install playwright
# !playwright install chromium

In [3]:
from __future__ import annotations

import json
import re
from pathlib import Path
from dataclasses import dataclass, asdict
from difflib import SequenceMatcher
from typing import Optional, List, Dict, Any, Tuple

import certifi
import pandas as pd
import requests
from bs4 import BeautifulSoup

SUNNAH_URL = "https://sunnah.com/hisn"
HADITHBD_URL = "https://hadithbd.com/books/fullbook/?book=4"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; hisn-merge-notebook/1.0; +https://example.com/bot)"
}

ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
BENGALI_RE = re.compile(r"[\u0980-\u09FF]")
SPACE_RE = re.compile(r"\s+")

SUNNAH_CHAPTER_RE = re.compile(r"^\((\d+)\)$")
SUNNAH_DUA_RE = re.compile(r"^Hisn al-Muslim (\d+)$")

HADITHBD_ENTRY_RE = re.compile(
    r"^(?P<num>[0-9০-৯]+)\s*-\s*(?:\((?P<num2>[0-9০-৯]+)\))?\s*(?P<rest>.*)$"
)
HADITHBD_CHAPTER_RE = re.compile(
    r"^(?P<chap>[0-9০-৯]+)\.\s*(?P<title>.+)$"
)

ARABIC_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ARABIC_PUNCT_RE = re.compile(r"[\[\]\(\)«»\"'.,;:!؟،\-—_]+")
TRANSLIT_CHARS = set("āīūḥḍṭẓṣʿʾ`'‘’")

BENGALI_DIGIT_MAP = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")


@dataclass
class SunnahChapter:
    chapter_number: int
    chapter_en: str
    chapter_ar: Optional[str] = None


@dataclass
class SunnahDua:
    dua_id: int
    chapter_number: Optional[int]
    chapter_en: Optional[str]
    chapter_ar: Optional[str]
    transliteration_en: Optional[str]
    english_translation: Optional[str]
    arabic: Optional[str]
    reference: Optional[str]
    notes: Optional[str]
    raw_block: List[str]


@dataclass
class HadithBDChapter:
    chapter_number_bn: int
    chapter_bn: str


@dataclass
class HadithBDDua:
    dua_id: int
    chapter_number_bn: Optional[int]
    chapter_bn: Optional[str]
    transliteration_bn: Optional[str]
    bengali_translation: Optional[str]
    arabic: Optional[str]
    references_bn: Optional[str]
    notes_bn: Optional[str]
    raw_block: List[str]

In [4]:
def fetch_html(url: str) -> str:
    verify: Any = certifi.where()
    try:
        import truststore

        truststore.inject_into_ssl()
        verify = True
    except ImportError:
        pass

    try:
        resp = requests.get(url, headers=HEADERS, timeout=45, verify=verify)
        resp.raise_for_status()
        return resp.text
    except requests.exceptions.SSLError as exc:
        raise RuntimeError(
            "SSL verification failed for hadithbd.com. In this notebook kernel, run `!pip install -U certifi truststore` and restart the kernel. Browsers often still work because they use the OS trust store, while this Python environment may be using an older CA bundle."
        ) from exc


def normalize_text(text: str) -> str:
    return SPACE_RE.sub(" ", text).strip()


def html_to_lines(html: str) -> List[str]:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    lines = [normalize_text(line) for line in text.splitlines()]
    return [line for line in lines if line]


def contains_bengali(text: str) -> bool:
    return bool(BENGALI_RE.search(text))


def arabic_ratio(text: str) -> float:
    if not text:
        return 0.0
    return len(ARABIC_RE.findall(text)) / max(len(text), 1)


def bengali_ratio(text: str) -> float:
    if not text:
        return 0.0
    return len(BENGALI_RE.findall(text)) / max(len(text), 1)


def is_probably_arabic(text: str) -> bool:
    return arabic_ratio(text) > 0.20


def is_probably_bengali(text: str) -> bool:
    return bengali_ratio(text) > 0.20


def translit_score(text: str) -> int:
    return sum(ch in TRANSLIT_CHARS for ch in text)


def bn_to_int(s: str) -> int:
    return int(s.translate(BENGALI_DIGIT_MAP))


def maybe_parse_bn_number(s: str) -> Optional[int]:
    try:
        return bn_to_int(s)
    except Exception:
        return None

## 1) Scrape Sunnah.com

In [5]:
def split_mixed_latin_line(line: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    line = normalize_text(line)
    if not line:
        return None, None, None

    if translit_score(line) == 0:
        return None, line, None

    parts = re.split(r'(?<=[\.\?\!])\s+(?=[A-Z"\(\[])', line)

    translit_parts = []
    english_parts = []
    switched = False

    for part in parts:
        part = normalize_text(part)
        if not part:
            continue
        if not switched and translit_score(part) >= 2:
            translit_parts.append(part)
        else:
            switched = True
            english_parts.append(part)

    return (
        " ".join(translit_parts).strip() or None,
        " ".join(english_parts).strip() or None,
        None,
    )


def find_sunnah_content_start(lines: List[str]) -> int:
    for i in range(len(lines) - 1):
        if SUNNAH_CHAPTER_RE.fullmatch(lines[i]) and lines[i + 1].startswith("Chapter:"):
            return i
    for i, line in enumerate(lines):
        if line == "Hisn al-Muslim 1":
            return i
    raise ValueError("Could not find Sunnah content start.")


SUNNAH_NOISE_LINES = {"Report Error", "|", "Share", "Copy", "▼", "Report Error | Share | Copy ▼"}


def is_sunnah_noise_line(line: str) -> bool:
    return normalize_text(line) in SUNNAH_NOISE_LINES


def is_sunnah_noise_block(block: List[str]) -> bool:
    cleaned = [normalize_text(line) for line in block if normalize_text(line)]
    return bool(cleaned) and all(is_sunnah_noise_line(line) for line in cleaned)


def parse_sunnah_dua_block(block: List[str]) -> Dict[str, Optional[str]]:
    arabic_lines = []
    ref_lines = []
    latin_lines = []
    notes_lines = []

    for line in block:
        if line.startswith("Reference"):
            ref_lines.append(line)
            continue
        if is_sunnah_noise_line(line):
            continue
        if is_probably_arabic(line):
            arabic_lines.append(line)
            continue
        latin_lines.append(line)

    translit_chunks = []
    english_chunks = []

    for line in latin_lines:
        t, e, n = split_mixed_latin_line(line)
        if t:
            translit_chunks.append(t)
        if e:
            english_chunks.append(e)
        if n:
            notes_lines.append(n)

    return {
        "transliteration_en": " ".join(translit_chunks).strip() or None,
        "english_translation": " ".join(english_chunks).strip() or None,
        "arabic": " ".join(arabic_lines).strip() or None,
        "reference": " ".join(ref_lines).strip() or None,
        "notes": " ".join(notes_lines).strip() or None,
    }


def parse_sunnah(lines: List[str]) -> Tuple[List[SunnahChapter], List[SunnahDua]]:
    start = find_sunnah_content_start(lines)
    lines = lines[start:]

    chapters: List[SunnahChapter] = []
    duas: List[SunnahDua] = []

    current_chapter: Optional[SunnahChapter] = None
    i = 0

    while i < len(lines):
        line = lines[i]

        m = SUNNAH_CHAPTER_RE.fullmatch(line)
        if m and i + 1 < len(lines) and lines[i + 1].startswith("Chapter:"):
            chapter_number = int(m.group(1))
            chapter_en = lines[i + 1].replace("Chapter:", "").strip()

            chapter_ar = None
            j = i + 2
            if j < len(lines) and SUNNAH_CHAPTER_RE.fullmatch(lines[j]):
                j += 1
            if j < len(lines) and is_probably_arabic(lines[j]):
                chapter_ar = lines[j]
                j += 1

            current_chapter = SunnahChapter(
                chapter_number=chapter_number,
                chapter_en=chapter_en,
                chapter_ar=chapter_ar,
            )
            chapters.append(current_chapter)
            i = j
            continue

        m = SUNNAH_DUA_RE.fullmatch(line)
        if m:
            dua_id = int(m.group(1))
            i += 1
            block = []

            while i < len(lines):
                next_line = lines[i]
                next_is_new_dua = SUNNAH_DUA_RE.fullmatch(next_line) is not None
                next_is_new_chapter = (
                    SUNNAH_CHAPTER_RE.fullmatch(next_line) is not None
                    and i + 1 < len(lines)
                    and lines[i + 1].startswith("Chapter:")
                )
                if next_is_new_dua or next_is_new_chapter:
                    break
                block.append(next_line)
                i += 1

            if is_sunnah_noise_block(block):
                continue

            parsed = parse_sunnah_dua_block(block)
            duas.append(
                SunnahDua(
                    dua_id=dua_id,
                    chapter_number=current_chapter.chapter_number if current_chapter else None,
                    chapter_en=current_chapter.chapter_en if current_chapter else None,
                    chapter_ar=current_chapter.chapter_ar if current_chapter else None,
                    transliteration_en=parsed["transliteration_en"],
                    english_translation=parsed["english_translation"],
                    arabic=parsed["arabic"],
                    reference=parsed["reference"],
                    notes=parsed["notes"],
                    raw_block=block,
                )
            )
            continue

        i += 1

    return chapters, duas

In [6]:
def get_node_text(node: Any) -> Optional[str]:
    if node is None:
        return None
    text = normalize_text(node.get_text("\n"))
    return text or None


def parse_sunnah_html(html: str) -> Tuple[List[SunnahChapter], List[SunnahDua]]:
    soup = BeautifulSoup(html, "html.parser")
    all_hadith = soup.select_one("div.AllHadith")
    if all_hadith is None:
        raise ValueError("Could not find Sunnah dua container.")

    chapters: List[SunnahChapter] = []
    duas: List[SunnahDua] = []
    current_chapter: Optional[SunnahChapter] = None

    for child in all_hadith.children:
        if getattr(child, "name", None) != "div":
            continue

        classes = set(child.get("class", []))

        if "chapter" in classes:
            chapter_number_text = get_node_text(child.select_one(".echapno")) or ""
            chapter_number_match = re.search(r"\d+", chapter_number_text)
            if chapter_number_match is None:
                continue

            current_chapter = SunnahChapter(
                chapter_number=int(chapter_number_match.group(0)),
                chapter_en=get_node_text(child.select_one(".englishchapter")) or "",
                chapter_ar=get_node_text(child.select_one(".arabicchapter")),
            )
            chapters.append(current_chapter)
            continue

        if not {"actualHadithContainer", "hadith_container_hisn"}.issubset(classes):
            continue

        dua_ref = get_node_text(child.select_one(".hadith_reference_sticky")) or ""
        m = SUNNAH_DUA_RE.fullmatch(dua_ref)
        if m is None:
            continue

        transliteration = get_node_text(child.select_one(".english_hadith_full .transliteration"))
        english_translation = get_node_text(child.select_one(".english_hadith_full .translation"))
        arabic_node = child.select_one(".arabic_hadith_full .arabic_text_details")
        if arabic_node is None:
            arabic_node = child.select_one(".arabic_hadith_full")
        arabic = get_node_text(arabic_node)
        reference = get_node_text(child.select_one(".english_hadith_full .hisn_english_reference"))

        raw_block = [
            value
            for value in [transliteration, english_translation, reference, arabic]
            if value
        ]

        duas.append(
            SunnahDua(
                dua_id=int(m.group(1)),
                chapter_number=current_chapter.chapter_number if current_chapter else None,
                chapter_en=current_chapter.chapter_en if current_chapter else None,
                chapter_ar=current_chapter.chapter_ar if current_chapter else None,
                transliteration_en=transliteration,
                english_translation=english_translation,
                arabic=arabic,
                reference=reference,
                notes=None,
                raw_block=raw_block,
            )
        )

    return chapters, duas


sunnah_html = fetch_html(SUNNAH_URL)
sunnah_chapters, sunnah_duas = parse_sunnah_html(sunnah_html)

print("Sunnah chapter count:", len(sunnah_chapters))
print("Sunnah dua count:", len(sunnah_duas))

sunnah_df = pd.DataFrame([asdict(d) for d in sunnah_duas])
sunnah_df.head(5)

Sunnah chapter count: 132
Sunnah dua count: 267


,dua_id,chapter_number,chapter_en,chapter_ar,transliteration_en,english_translation,arabic,reference,notes,raw_block
0,1,1,Chapter: When waking up,أذكار الاستيقاظ من النوم,Alḥamdu lillāhil-ladhī 'aḥyānā ba`da mā 'amāta...,Praise is to Allah Who gives us life after He ...,الحَمْـدُ لِلّهِ الّذي أَحْـيانا بَعْـدَ ما أَ...,"Al-Bukhari, cf. Al-Asqalani, Fathul-Bari 11/11...",None,[Alḥamdu lillāhil-ladhī 'aḥyānā ba`da mā 'amāt...
1,2,1,Chapter: When waking up,أذكار الاستيقاظ من النوم,"Lā 'ilāha 'illallāhu waḥdahu la sharīka lahu, ...",There is none worthy of worship but Allah alon...,لا إلهَ إلاّ اللّهُ وَحْـدَهُ لا شَـريكَ له، ل...,"Whoever says this will be forgiven, and if he ...",None,"[Lā 'ilāha 'illallāhu waḥdahu la sharīka lahu,..."
2,3,1,Chapter: When waking up,أذكار الاستيقاظ من النوم,"Al-ḥamdu lillāhil-ladhī `āfānī fī jasadī, wa r...",Praise is to Allah Who gave strength to my bod...,الحمدُ للهِ الذي عافاني في جَسَدي وَرَدّ عَليّ...,At-Tirmidhi 5/473. See Al-Albani's Sahih Tirmi...,None,"[Al-ḥamdu lillāhil-ladhī `āfānī fī jasadī, wa ..."
3,4,1,Chapter: When waking up,أذكار الاستيقاظ من النوم,'Inna fī khalqi-ssamāwāti wal-'arđi wakhtilāfi...,"Indeed, in the creation of the heavens and the...",إِنَّ فِي خَلْقِ السَّمَوَاتِ وَالأَرْضِ وَاخْ...,"Qur'an Āl-'Imran 3: 190-200; Al-Bukhari, cf. A...",None,['Inna fī khalqi-ssamāwāti wal-'arđi wakhtilāf...
4,5,2,Chapter: When wearing a garment,دعاء لبس الثوب,Alḥamdu lillāhil-ladhī kasānī hādhā (aththawba...,Praise is to Allah Who has clothed me with thi...,الحمدُ للهِ الّذي كَساني هذا (الثّوب) وَرَزَقَ...,"Al-Bukhari, Muslim, Abu Dawud, Ibn Majah, At-T...",None,[Alḥamdu lillāhil-ladhī kasānī hādhā (aththawb...


In [7]:
sunnah_df.to_csv("sunnah_duas.csv", index=False)

## 2) Scrape HadithBD fullbook

In [8]:
def is_hadithbd_chapter_header(line: str) -> bool:
    m = HADITHBD_CHAPTER_RE.fullmatch(line)
    if not m:
        return False
    title = m.group("title").strip()

    if "【" in line or "】" in line:
        return False
    if line.startswith("["):
        return False
    if "টি" in title:
        return False
    if HADITHBD_ENTRY_RE.fullmatch(line):
        return False

    return True


def find_hadithbd_content_start(lines: List[str]) -> int:
    for i, line in enumerate(lines):
        if is_hadithbd_chapter_header(line):
            return i
    raise ValueError("Could not find HadithBD content start.")


HADITHBD_NOISE_LINES = {
    "শেয়ার ও অন্যান্য",
    "শেয়ার ও অন্যান্য",
    "শেয়ার লিঙ্ক",
    "শেয়ার লিঙ্ক",
    "ইমেইল করুন",
    "ভুল পেলে রিপোর্ট করুন",
    "Close",
    "×",
    "আলাদা পেজে খুলুন",
    "কপি",
}


HADITHBD_PAGE_INFO_RE = re.compile(
    r"দেখানো হচ্ছেঃ\s*(?P<start>[0-9০-৯]+)\s*থেকে\s*(?P<end>[0-9০-৯]+)\s*পর্যন্ত,\s*সর্বমোট\s*(?P<total>[0-9০-৯]+)\s*টি রেকর্ড(?:ের মধ্য থেকে)?"
)


HADITHBD_INLINE_REF_RE = re.compile(r"\[(?P<num>[0-9০-৯]+)\]")
HADITHBD_REFERENCE_ID_RE = re.compile(r"^\[(?P<num>[0-9০-৯]+)\]")


def is_hadithbd_noise_line(line: str) -> bool:
    line = normalize_text(line)
    return (
        not line
        or line in HADITHBD_NOISE_LINES
        or line.startswith("وَصَلَّى اللَّهُ وَسَلَّمَ")
        or line.startswith("আল্লাহ্ দরূদ ও সালাম")
        or line.startswith("আল্লাহ দরূদ ও সালাম")
    )


def is_hadithbd_reference_line(line: str) -> bool:
    line = normalize_text(line)
    return line.startswith("[") and "]" in line[:5]


def is_hadithbd_bracket_note(line: str) -> bool:
    line = normalize_text(line)
    return line.startswith("[") and line.endswith("]") and not is_hadithbd_reference_line(line)


def is_hadithbd_subentry_start(line: str) -> bool:
    return re.match(r"^\([0-9০-৯]+\)\s*", normalize_text(line)) is not None


def strip_hadithbd_subentry_prefix(line: str) -> str:
    return re.sub(r"^\([0-9০-৯]+\)\s*", "", normalize_text(line)).strip()


def extract_hadithbd_parenthetical_text(line: str) -> Optional[str]:
    line = normalize_text(line)
    if not line.startswith("("):
        return None
    closing_idx = line.find(")")
    if closing_idx == -1:
        return None
    return line[: closing_idx + 1]


def is_hadithbd_translit_line(line: str) -> bool:
    if is_hadithbd_subentry_start(line):
        return False
    candidate = extract_hadithbd_parenthetical_text(line)
    if candidate is None:
        return False
    return is_probably_bengali(candidate)


def is_hadithbd_end_marker(line: str) -> bool:
    line = normalize_text(line)
    return (
        line.startswith("দেখানো হচ্ছেঃ")
        or line == "কপিরাইট"
        or line.startswith("2013-")
        or line.startswith("Developer:")
    )


def find_hadithbd_page_content_start(lines: List[str]) -> int:
    first_entry_index = None

    for i, line in enumerate(lines):
        if HADITHBD_ENTRY_RE.fullmatch(line):
            first_entry_index = i
            break

    if first_entry_index is None:
        raise ValueError("Could not find HadithBD page content start.")

    for i in range(first_entry_index, -1, -1):
        if is_hadithbd_chapter_header(lines[i]):
            return i

    return first_entry_index


def trim_hadithbd_page_lines(lines: List[str]) -> List[str]:
    start = find_hadithbd_page_content_start(lines)
    trimmed = []

    for line in lines[start:]:
        if is_hadithbd_end_marker(line):
            break
        if is_hadithbd_noise_line(line):
            continue
        trimmed.append(line)

    return trimmed


def extract_hadithbd_page_info(lines: List[str]) -> Optional[Tuple[int, int, int]]:
    for i in range(len(lines)):
        window = " ".join(normalize_text(line) for line in lines[i : i + 6])
        m = HADITHBD_PAGE_INFO_RE.search(window)
        if m:
            start = maybe_parse_bn_number(m.group("start"))
            end = maybe_parse_bn_number(m.group("end"))
            total = maybe_parse_bn_number(m.group("total"))
            if start and end and total:
                return start, end, total
    return None


def build_hadithbd_page_url(base_url: str, page_num: int, total: int) -> str:
    if page_num == 0:
        return base_url
    return f"{base_url}&pageNum_bookData={page_num}&totalRows_bookData={total}"


def fetch_all_hadithbd_pages(base_url: str) -> List[Tuple[int, str, str]]:
    first_html = fetch_html(base_url)
    first_lines = html_to_lines(first_html)
    page_info = extract_hadithbd_page_info(first_lines)
    pages: List[Tuple[int, str, str]] = [(0, base_url, first_html)]

    if page_info is None:
        return pages

    start, end, total = page_info
    page_size = max(end - start + 1, 1)
    total_pages = (total + page_size - 1) // page_size

    for page_num in range(1, total_pages):
        page_url = build_hadithbd_page_url(base_url, page_num, total)
        page_html = fetch_html(page_url)
        pages.append((page_num, page_url, page_html))

    return pages


def save_hadithbd_html_pages(pages: List[Tuple[int, str, str]], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    for page_num, page_url, page_html in pages:
        file_name = f"page_{page_num:02d}.html"
        (output_dir / file_name).write_text(page_html, encoding="utf-8")
        manifest.append({"page_num": page_num, "url": page_url, "file_name": file_name})

    (output_dir / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_hadithbd_html_pages(output_dir: Path) -> List[Tuple[int, str, str]]:
    manifest_path = output_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"HadithBD manifest not found: {manifest_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    pages = []
    for item in manifest:
        page_num = int(item["page_num"])
        page_url = item["url"]
        file_name = item["file_name"]
        page_html = (output_dir / file_name).read_text(encoding="utf-8")
        pages.append((page_num, page_url, page_html))

    return pages


def fetch_all_hadithbd_lines(base_url: str) -> List[str]:
    all_lines = []
    for _, _, page_html in fetch_all_hadithbd_pages(base_url):
        page_lines = trim_hadithbd_page_lines(html_to_lines(page_html))
        all_lines.extend(page_lines)
    return all_lines


def extract_hadithbd_inline_reference_ids(block: List[str]) -> List[int]:
    ref_ids = []

    for line in block:
        if is_hadithbd_reference_line(line):
            continue
        for m in HADITHBD_INLINE_REF_RE.finditer(line):
            ref_id = maybe_parse_bn_number(m.group("num"))
            if ref_id is not None:
                ref_ids.append(ref_id)

    return ref_ids


def parse_hadithbd_dua_block(block: List[str]) -> Dict[str, Optional[str]]:
    arabic_lines = []
    translit_bn_lines = []
    bengali_translation_lines = []
    reference_lines = []
    notes_lines = []

    first_bengali_translation_seen = False
    inline_reference_ids = set(extract_hadithbd_inline_reference_ids(block))
    in_reference_section = False
    current_reference_is_relevant = False

    for line in block:
        if is_hadithbd_noise_line(line):
            continue

        ref_match = HADITHBD_REFERENCE_ID_RE.match(normalize_text(line))
        if ref_match:
            in_reference_section = True
            ref_id = maybe_parse_bn_number(ref_match.group("num"))
            current_reference_is_relevant = not inline_reference_ids or ref_id in inline_reference_ids
            if current_reference_is_relevant:
                reference_lines.append(line)
            continue

        if in_reference_section:
            if current_reference_is_relevant:
                reference_lines.append(line)
            continue

        if is_hadithbd_bracket_note(line):
            notes_lines.append(line)
            continue

        if is_probably_arabic(line):
            arabic_lines.append(strip_hadithbd_subentry_prefix(line))
            continue

        if is_hadithbd_translit_line(line):
            translit_bn_lines.append(extract_hadithbd_parenthetical_text(line))
            continue

        if is_probably_bengali(line):
            cleaned = re.sub(r"^[0-9০-৯]+\s*-\s*(?:\([0-9০-৯]+\))?\s*", "", line).strip()
            cleaned = strip_hadithbd_subentry_prefix(cleaned)
            if not first_bengali_translation_seen or is_hadithbd_subentry_start(line):
                bengali_translation_lines.append(cleaned)
                first_bengali_translation_seen = True
            else:
                notes_lines.append(line)
            continue

        notes_lines.append(line)

    translit_bn = " ".join(translit_bn_lines).strip() or None
    if translit_bn:
        translit_bn = translit_bn.strip("()").strip()

    return {
        "arabic": " ".join(arabic_lines).strip() or None,
        "transliteration_bn": translit_bn,
        "bengali_translation": " ".join(bengali_translation_lines).strip() or None,
        "references_bn": " ".join(reference_lines).strip() or None,
        "notes_bn": " ".join(notes_lines).strip() or None,
    }


def parse_hadithbd(lines: List[str]) -> Tuple[List[HadithBDChapter], List[HadithBDDua]]:
    start = find_hadithbd_content_start(lines)
    lines = lines[start:]

    chapters: List[HadithBDChapter] = []
    duas: List[HadithBDDua] = []

    current_chapter: Optional[HadithBDChapter] = None
    current_dua_id: Optional[int] = None
    current_block: List[str] = []
    current_prelude_block: List[str] = []
    last_dua_id: Optional[int] = None

    def append_dua(dua_id: int, chapter: Optional[HadithBDChapter], block: List[str]) -> None:
        parsed = parse_hadithbd_dua_block(block)
        duas.append(
            HadithBDDua(
                dua_id=dua_id,
                chapter_number_bn=chapter.chapter_number_bn if chapter else None,
                chapter_bn=chapter.chapter_bn if chapter else None,
                transliteration_bn=parsed["transliteration_bn"],
                bengali_translation=parsed["bengali_translation"],
                arabic=parsed["arabic"],
                references_bn=parsed["references_bn"],
                notes_bn=parsed["notes_bn"],
                raw_block=block.copy(),
            )
        )

    def flush_current() -> None:
        nonlocal current_dua_id, current_block, last_dua_id
        if current_dua_id is None:
            return
        append_dua(current_dua_id, current_chapter, current_block)
        last_dua_id = current_dua_id
        current_dua_id = None
        current_block = []

    def flush_orphan_block() -> None:
        nonlocal current_prelude_block, last_dua_id
        if not current_prelude_block or current_chapter is None or last_dua_id is None:
            current_prelude_block = []
            return

        # HadithBD chapter 31 stores the bad-dream guidance as five unnumbered subitems.
        # Sunnah splits that content into dua_id 114 (items 1-4) and 115 (item 5).
        if "খারাপ স্বপ্ন" in current_chapter.chapter_bn:
            subentry_lines = [line for line in current_prelude_block if is_hadithbd_subentry_start(line)]
            if len(subentry_lines) >= 5:
                append_dua(last_dua_id + 1, current_chapter, subentry_lines[:4])
                last_dua_id += 1
                append_dua(last_dua_id + 1, current_chapter, subentry_lines[4:5])
                last_dua_id += 1
                current_prelude_block = []
                return

        append_dua(last_dua_id + 1, current_chapter, current_prelude_block)
        last_dua_id += 1
        current_prelude_block = []

    for line in lines:
        if is_hadithbd_noise_line(line):
            continue

        if is_hadithbd_chapter_header(line):
            flush_current()
            flush_orphan_block()
            m = HADITHBD_CHAPTER_RE.fullmatch(line)
            chap_num = maybe_parse_bn_number(m.group("chap"))
            chap_title = m.group("title").strip()
            current_chapter = HadithBDChapter(
                chapter_number_bn=chap_num,
                chapter_bn=chap_title,
            )
            chapters.append(current_chapter)
            continue

        m = HADITHBD_ENTRY_RE.fullmatch(line)
        if m:
            dua_id = maybe_parse_bn_number(m.group("num"))
            if current_dua_id is not None and dua_id == current_dua_id:
                rest = m.group("rest").strip()
                if rest:
                    current_block.append(rest)
                continue

            flush_current()
            if current_prelude_block and last_dua_id is not None and dua_id is not None and dua_id > last_dua_id + 1:
                flush_orphan_block()

            current_dua_id = dua_id
            if current_prelude_block:
                current_block.extend(current_prelude_block)
                current_prelude_block = []
            rest = m.group("rest").strip()
            if rest:
                current_block.append(rest)
            continue

        if current_dua_id is not None and is_hadithbd_subentry_start(line) and is_probably_arabic(line):
            flush_current()
            current_prelude_block = [line]
            continue

        if current_dua_id is not None:
            current_block.append(line)
            continue

        if current_chapter and (
            is_probably_arabic(line)
            or is_hadithbd_translit_line(line)
            or is_probably_bengali(line)
            or is_hadithbd_reference_line(line)
            or is_hadithbd_bracket_note(line)
            or is_hadithbd_subentry_start(line)
        ):
            current_prelude_block.append(line)

    flush_current()
    flush_orphan_block()
    duas.sort(key=lambda dua: dua.dua_id)
    return chapters, duas

In [9]:
#fetch hadithbd HTML
hadithbd_html_dir = Path("./hadithbdhtml")
hadithbd_html_dir.absolute().mkdir(exist_ok=True)
hadithbd_pages = fetch_all_hadithbd_pages(HADITHBD_URL)
save_hadithbd_html_pages(hadithbd_pages, hadithbd_html_dir)

print("Saved HadithBD HTML pages to:", hadithbd_html_dir.resolve())
print("HadithBD page count fetched:", len(hadithbd_pages))

Saved HadithBD HTML pages to: /Users/rumman/work/quran_ar_en_word_scrapping/python_scripts/hisnul_muslim_scrapper/hadithbdhtml
HadithBD page count fetched: 14


In [10]:
assert hadithbd_html_dir is not None
if "hadithbd_pages" not in globals():
    hadithbd_pages = load_hadithbd_html_pages(hadithbd_html_dir)

hadithbd_lines = []
for _, _, page_html in hadithbd_pages:
    hadithbd_lines.extend(trim_hadithbd_page_lines(html_to_lines(page_html)))
hadithbd_chapters, hadithbd_duas = parse_hadithbd(hadithbd_lines)

print("Loaded HadithBD HTML pages from:", hadithbd_html_dir.resolve())
print("HadithBD page count loaded:", len(hadithbd_pages))
print("HadithBD chapter count observed from parsed headers:", len(hadithbd_chapters))
print("HadithBD dua count parsed:", len(hadithbd_duas))

hadithbd_df = pd.DataFrame([asdict(d) for d in hadithbd_duas])
hadithbd_df.head(10)

Loaded HadithBD HTML pages from: /Users/rumman/work/quran_ar_en_word_scrapping/python_scripts/hisnul_muslim_scrapper/hadithbdhtml
HadithBD page count loaded: 14
HadithBD chapter count observed from parsed headers: 127
HadithBD dua count parsed: 267


,dua_id,chapter_number_bn,chapter_bn,transliteration_bn,bengali_translation,arabic,references_bn,notes_bn,raw_block
0,1,1,ঘুম থেকে জেগে উঠার সময়ের যিক্‌রসমূহ,আলহামদু লিল্লা-হিল্লাযী আহ্ইয়া-না- বা‘দা মা- আ...,"“হামদ-প্রশংসা আল্লাহ্‌র জন্য, যিনি (নিদ্রারূপ)...",«الْحَمْدُ لِلَّهِ الَّذِيْ أَحْيَانَا بَعْدَ ...,NaN,NaN,[«الْحَمْدُ لِلَّهِ الَّذِيْ أَحْيَانَا بَعْدَ...
1,2,1,ঘুম থেকে জেগে উঠার সময়ের যিক্‌রসমূহ,লা ইলা-হা ইল্লাল্লা-হু ওয়াহ্‌দাহূ লা- শারীকালা...,"“একমাত্র আল্লাহ ছাড়া কোনো হক্ব ইলাহ নেই, তাঁর ...",«لاَ إِلَهَ إِلاَّ اللَّهُ وَحْدَهُ لاَ شَريكَ...,NaN,NaN,[«لاَ إِلَهَ إِلاَّ اللَّهُ وَحْدَهُ لاَ شَريك...
2,3,1,ঘুম থেকে জেগে উঠার সময়ের যিক্‌রসমূহ,"আল্‌হামদু লিল্লা-হিল্লাযী ‘আ-ফা-নী ফী জাসাদী, ...","“সকল হামদ-প্রশংসা আল্লাহ্‌র জন্য, যিনি আমার দে...",«الْحَمْدُ لِلَّهِ الَّذِي عَافَانِي فِي جَسَد...,NaN,NaN,[«الْحَمْدُ لِلَّهِ الَّذِي عَافَانِي فِي جَسَ...
3,4,1,ঘুম থেকে জেগে উঠার সময়ের যিক্‌রসমূহ,ইন্না ফী খলকিস্ সামাওয়াতি ওয়াল আরদি ওয়াখতিলা-ফ...,"নিশ্চয় আসমানসমূহ ও যমীনের সৃষ্টিতে, রাত ও দিনে...",﴿اِنَّ فِيْ خَلْقِ السَّمٰوٰتِ وَالْاَرْضِ وَا...,"[4] সূরা আলে ইমরান ১৯০-২০০; বুখারী, ফাতহুল বার...",NaN,[﴿اِنَّ فِيْ خَلْقِ السَّمٰوٰتِ وَالْاَرْضِ وَ...
4,5,2,কাপড় পরিধানের দো‘আ,আল্‌হামদু লিল্লা-হিল্লাযী কাসানী হা-যা (আসসাওবা,“সকল হামদ-প্রশংসা আল্লাহ্‌র জন্য; যিনি আমাকে এ...,«الْحَمْدُ للَّهِ الَّذِي كَسَانِي هَذَا (الثّ...,[1] হাদীসটি নাসাঈ ব্যতীত সুনান গ্রন্থকারদের সব...,NaN,[«الْحَمْدُ للَّهِ الَّذِي كَسَانِي هَذَا (الث...
5,6,3,নতুন কাপড় পরিধানের দো‘আ,আল্লা-হুম্মা লাকাল-হামদু আনতা কাসাওতানীহি। আসআ...,“হে আল্লাহ্! আপনারই জন্য সকল হাম্‌দ-প্রশংসা। আ...,«اللَّهُمَّ لَكَ الْحَمْدُ أَنْتَ كَسَوْتَنِيه...,"[1] আবূ দাউদ, নং ৪০২০; তিরমিযী, নং ১৭৬৭; বাগভী...",NaN,[«اللَّهُمَّ لَكَ الْحَمْدُ أَنْتَ كَسَوْتَنِي...
6,7,4,অপরকে নতুন কাপড় পরিধান করতে দেখলে তার জন্য দো‘আ,তুবলী ওয়া ইয়ুখলিফুল্লা-হু তা‘আলা,"“তুমি পুরাতন করে ফেলবে, আর মহান আল্লাহ এর স্থল...",«تُبْلِي وَيُخْلِفُ اللَّهُ تَعَالَى».,NaN,NaN,"[«تُبْلِي وَيُخْلِفُ اللَّهُ تَعَالَى»., (তুবল..."
7,8,4,অপরকে নতুন কাপড় পরিধান করতে দেখলে তার জন্য দো‘আ,"ইলবাস জাদীদান, ওয়া ‘ইশ হামীদান, ওয়া মুত শাহীদান","“নতুন কাপড় পরিধান কর, প্রশংসিতরূপে দিনাতিপাত ক...",«اِلْبَسْ جَدِيداً وَعِشْ حَمِيداً وَمُتْ شَهِ...,"[2] সুনান ইবন মাজাহ ২/১১৭৮, নং ৩৫৫৮; বাগাওয়ী, ...",NaN,[«اِلْبَسْ جَدِيداً وَعِشْ حَمِيداً وَمُتْ شَه...
8,9,5,কাপড় খুলে রাখার সময় কী বলবে,বিসমিল্লাহ,“আল্লাহ্‌র নামে (খুলে রাখলাম)”[1]।,«بِسْمِ اللَّهِ».,"[1] তিরমিযী ২/৫০৫, নং ৬০৬, ও অন্যান্য। আরও দেখ...",NaN,"[«بِسْمِ اللَّهِ»., (বিসমিল্লাহ), “আল্লাহ্‌র ন..."
9,10,6,পায়খানায় প্রবেশের দো‘আ,[বিসমিল্লাহি] আল্লা-হুম্মা ইন্নী আ‘ঊযু বিকা মি...,“[আল্লাহ্‌র নামে।] হে আল্লাহ! আমি আপনার নিকট অ...,«[بِسْمِ اللَّهِ] اللَّهُمَّ إِنِّي أَعُوذُ بِ...,"[1] বুখারী ১/৪৫, নং ১৪২; মুসলিম ১/২৮৩, নং ৩৭৫।...",NaN,[«[بِسْمِ اللَّهِ] اللَّهُمَّ إِنِّي أَعُوذُ ب...


In [11]:
hadithbd_df.to_csv("hadithbd_duas.csv", index=False)

## 3) Save raw extractions

In [12]:
with open("sunnah_hisn_raw.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "source_url": SUNNAH_URL,
            "chapter_count": len(sunnah_chapters),
            "dua_count": len(sunnah_duas),
            "chapters": [asdict(c) for c in sunnah_chapters],
            "duas": [asdict(d) for d in sunnah_duas],
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open("hadithbd_hisn_raw.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "source_url": HADITHBD_URL,
            "chapter_count": len(hadithbd_chapters),
            "dua_count": len(hadithbd_duas),
            "chapters": [asdict(c) for c in hadithbd_chapters],
            "duas": [asdict(d) for d in hadithbd_duas],
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved raw extraction JSON files.")

Saved raw extraction JSON files.


In [ ]:
## Merge CSVs

## 4) Merge on `dua_id`

In [ ]:
def normalize_arabic(text: Optional[str]) -> str:
    if text is None or pd.isna(text):
        return ""
    if not isinstance(text, str):
        text = str(text)
    if not text:
        return ""
    text = ARABIC_DIACRITICS_RE.sub("", text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ـ", "")
    text = ARABIC_PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def arabic_match_status(a: Optional[str], b: Optional[str]) -> Dict[str, Any]:
    na = normalize_arabic(a)
    nb = normalize_arabic(b)

    if not na and not nb:
        return {"status": "both_missing", "score": None}
    if not na or not nb:
        return {"status": "one_missing", "score": None}
    if na == nb:
        return {"status": "exact_match", "score": 1.0}

    score = SequenceMatcher(None, na, nb).ratio()

    if score >= 0.98:
        status = "near_exact"
    elif score >= 0.90:
        status = "close_match"
    else:
        status = "mismatch"

    return {"status": status, "score": round(score, 4)}


merged_df = sunnah_df.merge(
    hadithbd_df,
    on="dua_id",
    how="outer",
    suffixes=("_sunnah", "_hadithbd"),
)

statuses = []
scores = []

for _, row in merged_df.iterrows():
    result = arabic_match_status(row.get("arabic_sunnah"), row.get("arabic_hadithbd"))
    statuses.append(result["status"])
    scores.append(result["score"])

merged_df["arabic_match_status"] = statuses
merged_df["arabic_match_score"] = scores

In [ ]:
preferred_cols = [
    "dua_id",
    "chapter_number",
    "chapter_en",
    "chapter_ar",
    "chapter_number_bn",
    "chapter_bn",
    "transliteration_en",
    "english_translation",
    "transliteration_bn",
    "bengali_translation",
    "arabic_sunnah",
    "arabic_hadithbd",
    "reference",
    "references_bn",
    "notes",
    "notes_bn",
    "arabic_match_status",
    "arabic_match_score",
]

final_cols = [c for c in preferred_cols if c in merged_df.columns]
merged_df = merged_df[final_cols + [c for c in merged_df.columns if c not in final_cols]]

merged_df.head(10)

## 5) Save merged files and audit

In [ ]:
merged_df.to_csv("hisn_merged.csv", index=False, encoding="utf-8-sig")
merged_df.to_json("hisn_merged.json", orient="records", force_ascii=False, indent=2)

audit_df = merged_df[
    [
        "dua_id",
        "chapter_number",
        "chapter_en",
        "chapter_bn",
        "arabic_sunnah",
        "arabic_hadithbd",
        "arabic_match_status",
        "arabic_match_score",
    ]
].copy()

audit_df.to_csv("hisn_arabic_audit.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- hisn_merged.csv")
print("- hisn_merged.json")
print("- hisn_arabic_audit.csv")

## 6) Validation checks

In [ ]:
print("Merge shape:", merged_df.shape)
print()
print("Arabic match status counts:")
print(merged_df["arabic_match_status"].value_counts(dropna=False))
print()
print("Missing Sunnah rows:", merged_df["english_translation"].isna().sum())
print("Missing HadithBD rows:", merged_df["bengali_translation"].isna().sum())

In [ ]:
mismatch_df = merged_df[merged_df["arabic_match_status"].isin(["mismatch", "one_missing"])].copy()
mismatch_df.head(30)

In [ ]:
sample_df = merged_df.sample(min(10, len(merged_df)), random_state=42)
sample_df[[
    c for c in [
        "dua_id",
        "chapter_en",
        "chapter_bn",
        "transliteration_en",
        "english_translation",
        "transliteration_bn",
        "bengali_translation",
        "arabic_match_status",
        "arabic_match_score",
    ] if c in sample_df.columns
]]